# Stage 6 — Development-only calibration and threshold governance

Stage 5 tuning is preserved as negative evidence. Its honest nested estimate did not improve the fixed Stage 4 XGBoost, so this notebook carries forward only that untuned candidate. The sealed final holdout is never scored or inspected here.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from creditscope.stage6 import generate_stage6_artifacts


## Locked nested procedure

The persisted Stage 4 folds are the five outer evaluation folds. Inside each outer-training partition, four-fold OOF probabilities compare no calibration, sigmoid, and isotonic calibration by Brier score. The selected strategy then uses only those inner OOF probabilities to minimize `5 × FN + FP` across the fixed 0.05–0.50 grid. Calibration and threshold are separate governed decisions.

In [ ]:
run_summary = generate_stage6_artifacts(PROJECT_ROOT)
display(pd.DataFrame([run_summary['honest_nested_policy_metrics']]))


## Calibration and threshold stability

The theoretical threshold is `1 / (1 + 5) ≈ 0.1667` only if probabilities are perfectly calibrated and FP/FN costs fully describe the decision. It is a reference, not a forced setting. Outer-fold variation reveals how stable the complete selection procedure is.

In [ ]:
choices = pd.read_csv(PROJECT_ROOT / 'reports/stage6/outer_fold_policy_choices.csv')
calibration = pd.read_csv(PROJECT_ROOT / 'reports/stage6/full_development_calibration_comparison.csv')
stability = pd.read_csv(PROJECT_ROOT / 'reports/stage6/threshold_stability.csv')
display(choices)
display(calibration)
display(stability)


## Full-development policy selection and freeze

Only after the honest nested evaluation, the complete development population selects one calibration method and one threshold. These full-development scores are selection evidence, not an unbiased estimate. The frozen policy cannot change after final-holdout access.

In [ ]:
key_thresholds = pd.read_csv(PROJECT_ROOT / 'reports/stage6/key_threshold_comparison.csv')
policy = json.loads((PROJECT_ROOT / 'reports/stage6/frozen_model_policy.json').read_text())
display(key_thresholds)
display(Markdown(f"**Frozen calibration:** `{policy['calibration']['method']}`  \
**Frozen threshold:** `{policy['decision_threshold']:.2f}`  \
**Holdout status:** sealed; Stage 7 protocol defined but not executed."))


## What remains for Stage 7

Stage 7 may load the frozen policy, fit it on all 800 development rows, score the 200-row holdout once, and report only the predefined metrics. It must not revise the model, features, preprocessing, calibration, or threshold after seeing holdout results.